# p53 Mutant MD Simulation with ESMFold API

This notebook predicts mutant structures using ESMFold API and runs MD simulations.

**No conda required** - uses pip only with OpenMM's built-in structure fixing.

In [ ]:
# @title 1. Install Dependencies
import sys
!{sys.executable} -m pip install -q requests numpy matplotlib openmm mdtraj
print("Installation complete!")

In [ ]:
# @title 2. Imports
import requests
import time
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

from openmm import *
from openmm.app import *
from openmm.unit import *

try:
    import mdtraj as md
    HAS_MDTRAJ = True
except ImportError:
    HAS_MDTRAJ = False

os.makedirs("structures", exist_ok=True)
os.makedirs("trajectories", exist_ok=True)
print("Setup complete!")

In [ ]:
# @title 3. ESMFold API

def predict_structure(sequence, max_retries=3):
    """Predict structure using ESMFold API."""
    url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    for attempt in range(max_retries):
        try:
            print(f"  ESMFold API call (attempt {attempt+1})...")
            r = requests.post(url, data=sequence, timeout=300)
            if r.status_code == 200:
                print("  Success!")
                return r.text
            print(f"  Error {r.status_code}, retrying...")
            time.sleep(30)
        except Exception as e:
            print(f"  {e}, retrying...")
            time.sleep(10)
    raise RuntimeError("ESMFold API failed")

print("ESMFold ready.")

In [ ]:
# @title 4. Structure Fixer (Pure Python, no pdbfixer needed)

def add_terminal_oxt(pdb_lines):
    """
    Add OXT atom to C-terminus and return fixed lines.
    This creates proper carboxylate geometry for the terminal.
    """
    atoms = [l for l in pdb_lines if l.startswith('ATOM')]
    
    # Find last residue atoms
    last_resnum = max(int(l[22:26]) for l in atoms)
    last_res_atoms = [l for l in atoms if int(l[22:26]) == last_resnum]
    
    # Get C, O, CA coordinates
    c_atom = next((l for l in last_res_atoms if l[12:16].strip() == 'C'), None)
    o_atom = next((l for l in last_res_atoms if l[12:16].strip() == 'O'), None)
    
    if not c_atom or not o_atom:
        return pdb_lines  # Can't fix, return as-is
    
    # Check if OXT exists
    if any(l[12:16].strip() == 'OXT' for l in last_res_atoms):
        return pdb_lines  # Already has OXT
    
    # Calculate OXT position
    cx, cy, cz = float(c_atom[30:38]), float(c_atom[38:46]), float(c_atom[46:54])
    ox, oy, oz = float(o_atom[30:38]), float(o_atom[38:46]), float(o_atom[46:54])
    
    # OXT is opposite O across C (carboxylate geometry)
    oxt_x, oxt_y, oxt_z = 2*cx - ox, 2*cy - oy, 2*cz - oz
    
    # Create OXT line
    serial = len(atoms) + 1
    oxt = f"ATOM  {serial:5d}  OXT {c_atom[17:20]} {c_atom[21]}{c_atom[22:26]}    {oxt_x:8.3f}{oxt_y:8.3f}{oxt_z:8.3f}  1.00  0.00           O\n"
    
    # Insert OXT after last atom, before TER
    result = []
    for line in pdb_lines:
        if line.startswith('TER') or line.startswith('END'):
            result.append(oxt)
        result.append(line)
    
    # Ensure TER and END exist
    if not any(l.startswith('TER') for l in result):
        result.append('TER\n')
    if not any(l.startswith('END') for l in result):
        result.append('END\n')
    
    return result


def fix_structure_for_openmm(input_path, output_path):
    """
    Fix ESMFold structure for OpenMM:
    1. Add OXT to C-terminus
    2. Add hydrogens using OpenMM
    3. Return topology and positions ready for simulation
    """
    print(f"Fixing structure: {input_path}")
    
    # Read and fix PDB
    with open(input_path) as f:
        lines = f.readlines()
    
    fixed_lines = add_terminal_oxt(lines)
    
    # Write intermediate fixed PDB
    with open(output_path, 'w') as f:
        f.writelines(fixed_lines)
    print(f"  Added OXT, saved to {output_path}")
    
    # Load with OpenMM
    pdb = PDBFile(output_path)
    
    # Use implicit solvent forcefield first (more forgiving for terminals)
    ff_implicit = ForceField('amber14-all.xml', 'implicit/gbn2.xml')
    
    # Create modeller and add hydrogens
    modeller = Modeller(pdb.topology, pdb.positions)
    print("  Adding hydrogens...")
    modeller.addHydrogens(ff_implicit, pH=7.0)
    
    # Save with hydrogens
    h_path = output_path.replace('.pdb', '_h.pdb')
    with open(h_path, 'w') as f:
        PDBFile.writeFile(modeller.topology, modeller.positions, f)
    print(f"  Added hydrogens, saved to {h_path}")
    
    return modeller.topology, modeller.positions

print("Structure fixer ready.")

In [ ]:
# @title 5. p53 Sequence

P53_FULL = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK"
    "SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE"
    "RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS"
    "SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP"
    "PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG"
    "GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
)

CORE_START, CORE_END = 94, 312
P53_CORE = P53_FULL[CORE_START-1:CORE_END]
print(f"p53 core domain: {len(P53_CORE)} residues ({CORE_START}-{CORE_END})")

In [ ]:
# @title 6. Configuration - EDIT THIS

# ========== EDIT THESE ==========
TARGET = "R175H"           # Cancer mutation
RESCUE = ["N239Y"]         # Rescue mutation(s)
PRODUCTION_NS = 1.0        # Simulation length (ns)
# ================================

ALL_MUTS = [TARGET] + RESCUE
NAME = "_".join(ALL_MUTS)

TIMESTEP = 2.0  # fs
EQ_STEPS = 50000         # 100 ps equilibration
PROD_STEPS = int(PRODUCTION_NS * 1e6 / TIMESTEP)

print(f"Mutant: {NAME}")
print(f"Production: {PRODUCTION_NS} ns ({PROD_STEPS} steps)")

In [ ]:
# @title 7. Generate Mutant & Predict Structure

# Apply mutations
seq = P53_CORE
print("Applying mutations:")
for mut in ALL_MUTS:
    wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]
    idx = pos - CORE_START
    assert seq[idx] == wt, f"Expected {wt} at {pos}, got {seq[idx]}"
    seq = seq[:idx] + mt + seq[idx+1:]
    print(f"  {mut}")

# Predict structure
print("\nPredicting structure...")
pdb_str = predict_structure(seq)

raw_path = f"structures/{NAME}_raw.pdb"
with open(raw_path, 'w') as f:
    f.write(pdb_str)
print(f"Saved: {raw_path}")

In [ ]:
# @title 8. Prepare for MD

fixed_path = f"structures/{NAME}_fixed.pdb"
topology, positions = fix_structure_for_openmm(raw_path, fixed_path)

# Now use explicit solvent forcefield
print("\nCreating solvated system...")
ff = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
modeller = Modeller(topology, positions)

print("  Adding solvent...")
modeller.addSolvent(ff, model='tip3p', padding=1.0*nanometer, ionicStrength=0.15*molar)

solv_path = f"structures/{NAME}_solvated.pdb"
with open(solv_path, 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)
print(f"  Saved: {solv_path}")

# Create system
print("  Creating system...")
system = ff.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometer,
    constraints=HBonds
)
print(f"\nSystem ready: {system.getNumParticles()} atoms")

In [ ]:
# @title 9. Energy Minimization

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
sim = Simulation(modeller.topology, system, integrator)
sim.context.setPositions(modeller.positions)

print("Minimizing...")
e0 = sim.context.getState(getEnergy=True).getPotentialEnergy()
sim.minimizeEnergy(maxIterations=1000)
e1 = sim.context.getState(getEnergy=True).getPotentialEnergy()
print(f"  {e0} -> {e1}")

positions = sim.context.getState(getPositions=True).getPositions()

In [ ]:
# @title 10. Equilibration (NPT)

system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
sim = Simulation(modeller.topology, system, integrator)
sim.context.setPositions(positions)
sim.context.setVelocitiesToTemperature(300*kelvin)

print(f"Equilibrating (100 ps)...")
sim.step(EQ_STEPS)

eq_path = f"structures/{NAME}_eq.pdb"
eq_pos = sim.context.getState(getPositions=True).getPositions()
with open(eq_path, 'w') as f:
    PDBFile.writeFile(sim.topology, eq_pos, f)
print(f"Saved: {eq_path}")

In [ ]:
# @title 11. Production MD

traj_path = f"trajectories/{NAME}.dcd"
log_path = f"trajectories/{NAME}.csv"

sim.reporters.append(DCDReporter(traj_path, 5000))
sim.reporters.append(StateDataReporter(log_path, 5000, step=True, time=True, 
    potentialEnergy=True, temperature=True, progress=True, 
    remainingTime=True, speed=True, totalSteps=PROD_STEPS))
sim.reporters.append(StateDataReporter(sys.stdout, 50000, step=True, time=True,
    potentialEnergy=True, temperature=True, progress=True,
    remainingTime=True, speed=True, totalSteps=PROD_STEPS))

print(f"Running {PRODUCTION_NS} ns production...\n")
sim.step(PROD_STEPS)
print(f"\nDone! Trajectory: {traj_path}")

In [ ]:
# @title 12. RMSD Analysis

if not HAS_MDTRAJ:
    print("Install mdtraj for analysis: pip install mdtraj")
else:
    traj = md.load(traj_path, top=eq_path)
    protein = traj.atom_slice(traj.topology.select('protein'))
    rmsd = md.rmsd(protein, protein, 0) * 10  # Angstrom
    t = np.arange(len(rmsd)) * TIMESTEP * 5000 / 1e6
    
    plt.figure(figsize=(10, 5))
    plt.plot(t, rmsd, 'b-', lw=0.5)
    plt.axhline(2, color='g', ls='--', label='Stable')
    plt.axhline(4, color='r', ls='--', label='Unstable')
    plt.xlabel('Time (ns)')
    plt.ylabel('RMSD (Å)')
    plt.title(f'{NAME} RMSD')
    plt.legend()
    plt.savefig(f"trajectories/{NAME}_rmsd.png", dpi=150)
    plt.show()
    
    print(f"\nMean: {rmsd.mean():.2f} Å, Final: {rmsd[-1]:.2f} Å")
    if rmsd[-1] < 2.5:
        print("STABLE")
    elif rmsd[-1] < 4:
        print("MODERATE")
    else:
        print("UNSTABLE")